In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import TimestampType, DateType
from delta.tables import DeltaTable

In [0]:
CATALOG = "project"
SCHEMA  = "silver"
DIM_TBL = "dim_customers"

BRONZE_CATALOG = "project"
BRONZE_SCHEMA  = "bronze"
BRONZE_TBL     = "customers"  # bronze source table name (append-only)

PK_COL = "customer_id"   # natural/business key

# Columns that define SCD2 changes (Type 2 tracked fields)
SCD2_COLUMNS = ["email","city","state"]

# Columns that define SCD3 changes (Type 3 tracked fields)
SCD3_COLUMNS = ["first_name","last_name"]

# All columns
BUSINESS_COLS = [PK_COL] + SCD2_COLUMNS + SCD3_COLUMNS

# Column in bronze that orders the latest record per PK
EVENT_TS_COL = "ingesttime"

# Far-future date for open SCD2 records
FUTURE_DATE = None

# Stream checkpoint location
CHECKPOINT_PATH = "/Volumes/project/_ops/streaming/customers/"


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.{DIM_TBL} (
  customer_skey BIGINT GENERATED ALWAYS AS IDENTITY,
  {PK_COL}       STRING,
  first_name      STRING,
  last_name       STRING,
  email          STRING,
  city           STRING,
  state          STRING,
  effective_date DATE,
  end_date       DATE,
  active_flag    BOOLEAN,
  last_seen_date DATE
) USING DELTA
""")

In [0]:
def build_latest_per_pk(df):
    """
    - Select only BUSINESS_COLS + EVENT_TS_COL from the incoming microbatch.
    - Cast EVENT_TS_COL to timestamp.
    - Keep only latest row per PK within this microbatch.
    """
    selected_cols = df.select(*[F.col(c) for c in BUSINESS_COLS], F.col(EVENT_TS_COL).cast(TimestampType()).alias(EVENT_TS_COL))

    w = Window.partitionBy(PK_COL).orderBy(F.col(EVENT_TS_COL).desc(), F.monotonically_increasing_id().desc())
    
    src_latest = selected_cols.withColumn("rnk", F.row_number().over(w)).filter(F.col("rnk")==1).drop("rnk")

    return src_latest

In [0]:
def add_scd2_hash(src_latest):
    """
    Add a normalized sha256 hash over SCD2_COLUMNS.
    """
    norm_expr = [F.coalesce(F.trim(F.upper(F.col(COL))),F.lit("__NULL__")) for COL in SCD2_COLUMNS]
    src_h = src_latest.withColumn("hash_256", F.sha2(F.concat_ws("||", *norm_expr),256))
    return src_h

In [0]:
def get_target_active_with_hash():
    """
    Read active rows from dim_customers and compute SCD2 hash for comparison.
    Returns df with columns: [PK_COL, t_hash]
    """
    tgt_active = spark.read.table(f"{CATALOG}.{SCHEMA}.{DIM_TBL}").where(F.col("active_flag")==True)
    norms_expr = [F.coalesce(F.trim(F.upper(F.col(COL))),F.lit("__NULL__")) for COL in SCD2_COLUMNS]
    tgt_h = tgt_active.withColumn("t_hash", F.sha2(F.concat_ws("||",*norms_expr), 256)).select(PK_COL, "t_hash")
    return tgt_h

In [0]:
def build_staged_changes(microbatch_df):
    """
    From raw microbatch -> latest per PK -> SCD2 hash -> staged with flags.
    """
    src_latest = build_latest_per_pk(microbatch_df)
    src_h = add_scd2_hash(src_latest)
    tgt_h = get_target_active_with_hash()
    
    joined = src_h.join(tgt_h, on=PK_COL, how="left")

    staged = (
        joined
        # as_of_date based on event timestamp (or use F.current_date() if you prefer)
        .withColumn("as_of_date", F.col(EVENT_TS_COL)).cast(DateType())
        .withColumn("is_new",      F.col("t_hash").isNull())
        .withColumn("is_changed",  F.col("t_hash").isNotNull() & (F.col("t_hash") != F.col("hash_256")))
        .withColumn("is_nochange", F.col("t_hash").isNotNull() & (F.col("t_hash") == F.col("hash_256")))
        .select(
            *[F.col(c) for c in BUSINESS_COLS],
            "as_of_date",
            "is_new",
            "is_changed",
            "is_nochange",
        )
    )
    return staged


In [0]:
def run_scd2_merges(staged_df):
    """
    Apply SCD2 logic into project.silver.dim_customers.

    Assumes staged_df has:
      - BUSINESS_COLS
      - as_of_date
      - is_new, is_changed, is_nochange

    Semantics:
      - One active row per customer_id (end_date IS NULL, active_flag = true)
      - When SCD2 columns change:
          * close old row (end_date = as_of_date, active_flag = false)
          * insert new active row with NULL end_date
    """

    # short-circuit: nothing to do
    if staged_df.limit(1).count() == 0:
        return

    delta_tbl = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA}.{DIM_TBL}")

    # ---------- MERGE #1: close changed rows & refresh last_seen ----------
    using_update = staged_df.select(
        *BUSINESS_COLS,
        "as_of_date",
        "is_changed",
        "is_nochange"
    )

    (delta_tbl.alias("T")
      .merge(
          using_update.alias("S"),
          f"T.{PK_COL} = S.{PK_COL} AND T.active_flag = true"
      )
      .whenMatchedUpdate(
          condition="S.is_changed",
          set={
              "end_date":       "S.as_of_date",
              "active_flag":    "false",
              "last_seen_date": "S.as_of_date",
          }
      )
      .whenMatchedUpdate(
          condition="S.is_nochange",
          set={
              "last_seen_date": "S.as_of_date",
          }
      )
      .execute()
    )

    # ---------- MERGE #2: insert new active versions for NEW + CHANGED ----------
    using_insert = staged_df.select(
        *BUSINESS_COLS,
        "as_of_date",
        "is_new",
        "is_changed"
    )

    (delta_tbl.alias("T")
      .merge(
          using_insert.alias("S"),
          f"T.{PK_COL} = S.{PK_COL} AND T.active_flag = true"
      )
      .whenNotMatchedInsert(
          condition="S.is_new OR S.is_changed",
          values={
              # business columns (customer_skey is identity, not included)
              **{c: f"S.{c}" for c in BUSINESS_COLS},
              "effective_date": "S.as_of_date",
              "end_date":       "CAST(NULL AS DATE)",        # NULL marks active row
              "active_flag":    "true",
              "last_seen_date": "S.as_of_date",
          }
      )
      .execute()
    )


In [0]:
def scd2_merge_customers_foreach_batch(microbatch_df, batch_id) -> None:
    """
    foreachBatch entrypoint for the customers dimension:
      1) Build staged changes
      2) Apply SCD2 MERGE logic
    """
    staged = build_staged_changes(microbatch_df)
    run_scd2_merges(staged)


In [0]:
customers_stream = (
    spark.readStream
         .table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TBL}")
)

query = (
    customers_stream.writeStream
        .option("checkpointLocation", CHECKPOINT_PATH)
        .foreachBatch(scd2_merge_customers_foreach_batch)
        .trigger(availableNow=True)   # or "availableNow" for one-shot
        .start()
)

In [0]:
# spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TBL}").printSchema()
# 